# VISTA: shared detector and evidence audit

**Status:** The project uses the bundled, pretrained `yolo26s.pt` checkpoint for every traffic video. We have **not fine-tuned a detector** and do not have a training dataset or trained `best.pt`. `yolov8n.pt` is an optional baseline used for comparison. This notebook documents what is actually implemented and measured.

The scene preset changes the counting line, stop zone, and confidence threshold for a camera. It does **not** change the model weights. Road-defect observations are marked by a reviewer; VISTA has no pothole detector.


In [1]:
from pathlib import Path
import sys, json, hashlib
from collections import Counter
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

def find_project():
    for parent in (Path.cwd(), *Path.cwd().parents):
        for candidate in (parent, parent / "phase2_prototype"):
            if (candidate / "yolo26s.pt").is_file():
                return candidate.resolve()
    raise FileNotFoundError("Open this notebook from the VISTA repository")

ROOT = find_project()
sys.path.insert(0, str(ROOT))
from web_app import SOURCE_DATA, SCENE_PRESETS
from video_pipeline import VEHICLE_CLASSES
print("Project folder:", ROOT.name)
print("Vehicle classes:", VEHICLE_CLASSES, "(car, motorcycle, bus, truck)")


Project folder: phase2_prototype
Vehicle classes: (2, 3, 5, 7) (car, motorcycle, bus, truck)


## 1. Checkpoint provenance

These are pretrained Ultralytics checkpoints bundled with VISTA. Hashes identify the exact files used by this notebook and the saved runs. A checkpoint file alone is not evidence of fine-tuning.


In [2]:
for name in ("yolo26s.pt", "yolov8n.pt"):
    path = ROOT / name
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    print(f"{name}: {path.stat().st_size / 1_000_000:.2f} MB · SHA-256 {digest}")

model = YOLO(str(ROOT / "yolo26s.pt"))  # one shared checkpoint for every scene below
print("Detector loaded once:", "yolo26s.pt")


yolo26s.pt: 20.42 MB · SHA-256 646f8bc3fe0a656803d95c294f7852321748cb29d13466a1af8862e2db384a1b
yolov8n.pt: 6.55 MB · SHA-256 f59b3d833e2ff32e194b5bb8e08d211dc7c5bdf144b90d2c8412c47ccfc83b36
Detector loaded once: yolo26s.pt


## 2. The same detector on six inputs

One instance of `yolo26s.pt` processes a middle frame from each included video. These are **single-frame sanity checks**, not traffic-event accuracy measurements. The main Python pipeline adds BoT-SORT tracking and the scene-specific event geometry after detection.


In [3]:
frames, keys = [], []
for key, (_title, filename, _source_note) in SOURCE_DATA.items():
    capture = cv2.VideoCapture(str(ROOT / "sample_data" / filename))
    if not capture.isOpened():
        raise FileNotFoundError(filename)
    capture.set(cv2.CAP_PROP_POS_FRAMES, int(capture.get(cv2.CAP_PROP_FRAME_COUNT) // 2))
    ok, frame = capture.read()
    capture.release()
    if not ok:
        raise ValueError(f"Cannot read a middle frame from {filename}")
    keys.append(key)
    frames.append(frame)

predictions = model.predict(frames, classes=list(VEHICLE_CLASSES), conf=0.25,
                            imgsz=640, device="cpu", verbose=False)
for key, prediction in zip(keys, predictions):
    classes = Counter(model.names[int(value)] for value in prediction.boxes.cls.tolist())
    print(f"{key:10s} {len(prediction.boxes):2d} vehicle boxes  {dict(classes)}")


signal      4 vehicle boxes  {'car': 4}
overhead    1 vehicle boxes  {'car': 1}
cuttack    15 vehicle boxes  {'car': 6, 'bus': 7, 'truck': 2}
bengaluru  13 vehicle boxes  {'car': 12, 'bus': 1}
pothole    12 vehicle boxes  {'car': 12}
stop        2 vehicle boxes  {'car': 2}


In [4]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7.5), constrained_layout=True)
for axis, key, prediction in zip(axes.flat, keys, predictions):
    axis.imshow(cv2.cvtColor(prediction.plot(), cv2.COLOR_BGR2RGB))
    axis.set_title(f"{key} · {len(prediction.boxes)} boxes", fontsize=11)
    axis.axis("off")
output = ROOT / "notebooks" / "outputs" / "shared_detector_six_scenes.png"
output.parent.mkdir(exist_ok=True)
fig.savefig(output, dpi=130)
plt.close(fig)
print("Saved visual audit:", output.relative_to(ROOT))


Saved visual audit: notebooks/outputs/shared_detector_six_scenes.png


![The same detector checkpoint applied to one frame from each of six included videos](outputs/shared_detector_six_scenes.png)

**Interpretation:** Boxes in this montage are detections only. The displayed box counts do not measure crossing accuracy, congestion, road speed, or delay.


## 3. Camera settings are not separate models

The same detector and BoT-SORT tracker are used for traffic videos. `SCENE_PRESETS` only selects a normalized line and zone for each camera. Uploaded videos use the user's chosen geometry. Track IDs restart for each independent run.


In [5]:
for key in keys:
    axis, line_at, zone, confidence = SCENE_PRESETS[key]
    print(f"{key:10s} line={axis}@{line_at:.2f}  zone={zone}  confidence={confidence:.2f}")


signal     line=y@0.60  zone=(0.45, 0.05, 1.0, 1.0)  confidence=0.25
overhead   line=y@0.55  zone=(0.02, 0.05, 0.98, 0.94)  confidence=0.25
cuttack    line=x@0.50  zone=(0.05, 0.25, 0.95, 0.95)  confidence=0.25
bengaluru  line=y@0.65  zone=(0.05, 0.4, 0.6, 0.95)  confidence=0.25
pothole    line=y@0.65  zone=(0.1, 0.45, 0.72, 0.95)  confidence=0.25
stop       line=y@0.55  zone=(0.02, 0.05, 0.98, 0.94)  confidence=0.10


## 4. Recompute the measured event comparison

The manually reviewed labels cover **11 line crossings in two short real clips**. Each saved run contains timestamped predictions. Matching requires the same event type, direction, and vehicle class within ±1 second. This is a small audit, not general model accuracy. India scene event totals and real stop events are not fully labeled.


In [6]:
from benchmark import CASES
from evaluate_run import read_labels
from evaluation import evaluate_by_type

totals = {"YOLOv8n baseline": [0, 0, 0], "YOLO26s primary": [0, 0, 0]}
for scene, label_file, run_ids in CASES:
    labels = read_labels(ROOT / "evaluation_data" / label_file)
    for name, run_id in run_ids.items():
        summary = json.loads((ROOT / "runs" / run_id / "summary.json").read_text())
        score = evaluate_by_type(summary["events"], labels)["crossing"]
        for index, field in enumerate(("tp", "fp", "fn")):
            totals[name][index] += score[field]
        print(f"{scene:15s} {name:17s} TP={score['tp']} FP={score['fp']} FN={score['fn']}")
for name, (tp, fp, fn) in totals.items():
    print(f"OVERALL {name:17s} matched={tp}/{tp+fn}  extra={fp}")


Overhead road   YOLOv8n baseline  TP=2 FP=0 FN=2
Overhead road   YOLO26s primary   TP=4 FP=0 FN=0
Traffic signal  YOLOv8n baseline  TP=7 FP=0 FN=0
Traffic signal  YOLO26s primary   TP=7 FP=0 FN=0
OVERALL YOLOv8n baseline  matched=9/11  extra=0
OVERALL YOLO26s primary   matched=11/11  extra=0


## 5. Road observations are a separate manual workflow

Vehicle detection does not find potholes. A reviewer draws a box on a photo or video frame and records source, date, place, and a follow-up. The recurrence check requires distinct media on distinct dates at a precise location.


In [7]:
from observations import load_records
for record in load_records(ROOT / "observations"):
    print(record["id"], "·", record["category"], "·", record["source_kind"],
          "· precise location:", record.get("precise_location", False))


OBS-KUMASI-001 · Pothole / broken surface · Public reference · precise location: False
OBS-KOLKATA-001 · Pothole / broken surface · Public reference · precise location: False


## What a genuine fine-tuning claim would require

To fine-tune **one shared vehicle detector** responsibly, first collect or license vehicle bounding-box annotations, split training and validation by camera/date, train a new checkpoint, save the training configuration and `best.pt`, then compare that checkpoint with the current pretrained model on untouched traffic clips and the same event-label audit. None of those training artifacts exists in this repository yet. Do not describe the current weights as team-trained during review.
